## Load the Last.fm Dataset

In [2]:
from implicit.datasets.lastfm import get_lastfm

artists, users, artist_user_plays = get_lastfm()

c:\Users\pouya\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Peek Inside the Last.fm Data

In [3]:
print(f"Artists: {len(artists):,}")
print(f"User: {len(users):,}")
print(f"Matrix shape (artists x users): {artist_user_plays.shape}")
print(f"Non-zero interactions: {artist_user_plays.nnz:,}")

Artists: 292,385
User: 358,868
Matrix shape (artists x users): (292385, 358868)
Non-zero interactions: 17,535,606


## Train-Test Split

In [4]:
from implicit.evaluation import train_test_split

# Transpose to (users x artists) format | rows -> users | columns -> items
user_plays = artist_user_plays.T.tocsr()

train_raw, test_raw = train_test_split(user_plays,
                                       train_percentage=0.8,
                                       random_state=42)

print(f"Train shape: {train_raw.shape}, non-zero: {train_raw.nnz:,}")
print(f"Test shape: {test_raw.shape}, non-zero: {test_raw.nnz:,}")

Train shape: (358868, 292385), non-zero: 14,028,047
Test shape: (358868, 292385), non-zero: 3,507,558


## Preprocess Data for Training

- **`bm25_weight`**: Applies BM25 scoring to user-item interactions, reducing the influence of popular items.

- **`csr_matrix`**: A sparse matrix format (Compressed Sparse Row) that stores only non-zero values efficiently.

```python
# Dense matrix (wastes space)
dense = [
    [0, 0, 5],
    [3, 0, 0],
    [0, 2, 0]
]

# CSR format (stores only non-zero)
from scipy.sparse import csr_matrix
sparse = csr_matrix(dense)

print(sparse.data)    # [5, 3, 2] - values
print(sparse.indices) # [2, 0, 1] - columns  
print(sparse.indptr)  # [0, 1, 2, 3] - row starts

## BM25 Weighting for ALS

In [5]:
from implicit.nearest_neighbours import bm25_weight

# ALS: apply BM25 weighting on train_raw
weighted_train_als = bm25_weight(train_raw,     # Sparse matrix (artists x users) with play counts
                                 K1=100,     # Saturation: higher = more weight for repeated plays
                                 B=0.8).tocsr()      # Normalization: higher = penalizes very active users more)

test_als = test_raw.sign()

## Train ALS Model

**Confidence Weight**  
`Confidence = 1 + α × (interaction strength)`

Higher confidence = more important to the model.  
Common `α` values: 1.0 to 40.0.

| Platform | Action | Weight |
|----------|--------|--------|
| E-commerce | Purchase | 10.0 |
| E-commerce | Add to cart | 5.0 |
| E-commerce | Click | 1.0 |
| E-commerce | View | 0.5 |
| YouTube | Watch time (minutes) | 1.0 - 5.0 (scaled) |
| YouTube | Like | 10.0 |
| YouTube | Subscribe | 20.0 |

In [6]:
from implicit.als import AlternatingLeastSquares

# Initialize ALS model
als_model = AlternatingLeastSquares(factors=128,                  # Latent factor dimension (embedding size)
                                    regularization=0.01,          # Prevents overfitting
                                    alpha=20.0,                   # confidence weight for positive interactions (plays/listens)
                                    iterations=50)                # Training epochs


# Train the model
als_model.fit(user_items=weighted_train_als,
              show_progress=True)

c:\Users\pouya\AppData\Local\Programs\Python\Python39\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [01:55<00:00,  2.32s/it]


## Binary Conversion for BPR and LMF

In [7]:
# converts non-zero to 1
train_binary = train_raw.sign()
test_binary = test_raw.sign()

## Train BPR Model

In [8]:
from implicit.bpr import BayesianPersonalizedRanking

bpr_model = BayesianPersonalizedRanking(factors=100,
                                        learning_rate=0.01,
                                        regularization=0.01,
                                        iterations=100,
                                        verify_negative_samples=True)

bpr_model.fit(user_items=train_binary,
              show_progress=True)

100%|██████████| 100/100 [03:25<00:00,  2.05s/it, train_auc=95.27%, skipped=1.75%]


## Train LMF Model

In [9]:
from implicit.lmf import LogisticMatrixFactorization

lmf_model = LogisticMatrixFactorization(factors=30,
                                        learning_rate=1.0,
                                        regularization=0.01,
                                        iterations=30)

lmf_model.fit(user_items=train_binary,
              show_progress=True)

100%|██████████| 30/30 [00:38<00:00,  1.28s/it]


## Models Evaluation

### 🎯 Binary Evaluation (Hits only)

| | |
|---|---|
| **Question** | "Did the user interact with this item at all?" |
| **Best for** | Discovery, new user onboarding, CTR optimization |
| **Fair to** | ✅ BPR / LMF (native format) <br> ⚠️ ALS (slightly handicapped) |

---

### ⚡ Weighted Evaluation (Engagement strength)

| | |
|---|---|
| **Question** | "Did the user strongly engage with this item?" |
| **Best for** | Retention, watch time, play-count-aware recommendations |
| **Fair to** | ✅ ALS (native format) <br> ❌ BPR / LMF (heavily handicapped) |

---

### 📈 Quick Summary

| Strategy | Best For | Fair To |
|----------|----------|---------|
| **Binary** | Discovery, CTR | BPR / LMF |
| **Weighted** | Retention, Engagement | ALS |

---

### 🎯 Final Takeaway

> **For predicting *whether* a user will listen (discovery) → BPR works best.**  
> **For predicting *how much* they'll listen (engagement) → ALS is the right choice.**

**Different questions. Different models. Both valid.**

### Evaluating ALS Model


In [10]:
from implicit.evaluation import ranking_metrics_at_k

als_metrics = ranking_metrics_at_k(als_model, weighted_train_als, test_als, K=10)

100%|██████████| 358511/358511 [03:29<00:00, 1713.42it/s]


### Evaluating BPR Model

In [11]:
bpr_metrics = ranking_metrics_at_k(bpr_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:23<00:00, 1759.05it/s]


### Evaluating LMF Model

In [12]:
lmf_metrics = ranking_metrics_at_k(lmf_model, train_binary, test_binary, K=10)

100%|██████████| 358511/358511 [03:20<00:00, 1788.96it/s]


## Model Performance Comparison

### Metrics Explained

| Metric | Question it answers | Example |
|--------|---------------------|---------|
| **Precision@10** | "Out of 10 recs, how many did user actually like?" | 0.054 = 5.4% → ~5 good recs out of 100 (sparse dataset) |
| **MAP** | "Are the good recs at the TOP of the list?" | 0.02 = low but expected for Last.fm |
| **NDCG** | "Does ranking order make sense? (valuing top positions more)" | 0.05 = model learns something, not random |
| **AUC** | "Given one liked + one disliked, does model pick the liked one?" | 0.50 = coin flip. 0.52 = barely better (normal for sparse data) |


**Why ALS wins for Last.fm:**

| Aspect | ALS | BPR | LMF |
|--------|:---:|:---:|:---:|
| Precision | 5.4% | 6.4% | 3.3% |
| **Play counts matter?** | ✅ Yes | ❌ No | ❌ No |
| **Fair eval on weighted** | ✅ | ❌ | ❌ |

**Final verdict:**
> BPR has higher precision, but ALS answers the **right question** for Last.fm:  
> *"How much will they listen?"* not *"Will they listen at all?"*

In [13]:
import pandas as pd

df = pd.DataFrame([als_metrics, bpr_metrics, lmf_metrics], index=['ALS', 'BPR', 'LMF'])

df.round(4)

,precision,map,ndcg,auc
ALS,0.0543,0.0206,0.0533,0.5242
BPR,0.0646,0.0281,0.0683,0.5289
LMF,0.0378,0.0143,0.0376,0.5168


## Training Final ALS Model on Full Data
Now that we've validated the model works, we train on **100% of the data** for maximum performance.

In [16]:
from implicit.nearest_neighbours import bm25_weight
from implicit.als import AlternatingLeastSquares

user_plays = artist_user_plays.T.tocsr()

user_plays_weighted = bm25_weight(user_plays, K1=100, B=0.8).tocsr()

# Initialize ALS model
final_als_model = AlternatingLeastSquares(factors=128,           # Latent factor dimension (embedding size)
                                          regularization=0.01,   # Prevents overfitting
                                          alpha=20.0,            # confidence weight for positive interactions (plays/listens)
                                          iterations=50)         # Training epochs


# Train the model
final_als_model.fit(user_items=user_plays_weighted,
                    show_progress=True,)

100%|██████████| 50/50 [02:18<00:00,  2.76s/it]


## Examine Learned Factors


**What the numbers mean:**

- Each user has **128 numbers** representing their "taste vector"
- Each artist has **128 numbers** representing their "vibe vector"
- **Similar taste vectors × similar vibe vectors = high recommendation score**

In [83]:
print(f"User factors shape: {final_als_model.user_factors.shape}")  # (n_users, factors)
print(f"Item factors shape: {final_als_model.item_factors.shape}")  # (n_items, factors)

User factors shape: (358868, 128)
Item factors shape: (292385, 128)


### Where `user_factors` comes from

**Only matrix factorization models** (like ALS) have `user_factors`.

#### What it is:
- After training, ALS learns two matrices:
  - `user_factors` - taste vector for each user
  - `item_factors` - vibe vector for each artist

## Example:
```python
model.user_factors.shape  # (n_users, n_factors)
model.user_factors[326348]  # User 326348's taste vector

In [92]:
similar_ids, scores = final_als_model.similar_items(107225, N=50)
for aid, score in zip(similar_ids, scores):
    print(f"{artists[aid]}: {score:.4f}")

eminem: 1.0000
kanye west: 0.9996
guns n roses: 0.9995
ac/dc: 0.9995
foo fighters: 0.9995
incubus: 0.9994
a perfect circle: 0.9994
tool: 0.9994
oasis: 0.9994
fatboy slim: 0.9993
koЯn: 0.9993
guns n' roses: 0.9993
garbage: 0.9993
marilyn manson: 0.9993
the killers: 0.9992
arctic monkeys: 0.9992
britney spears: 0.9992
snow patrol: 0.9992
the offspring: 0.9992
tenacious d: 0.9991
justin timberlake: 0.9991
audioslave: 0.9991
the kooks: 0.9991
my chemical romance: 0.9990
franz ferdinand: 0.9990
kaiser chiefs: 0.9990
pendulum: 0.9990
röyksopp: 0.9990
led zeppelin: 0.9990
the strokes: 0.9989
hans zimmer: 0.9989
jimi hendrix: 0.9989
black sabbath: 0.9989
pearl jam: 0.9989
keane: 0.9989
nelly furtado: 0.9989
deftones: 0.9989
him: 0.9989
blink-182: 0.9989
black eyed peas: 0.9989
bad religion: 0.9989
john mayer: 0.9989
avril lavigne: 0.9988
interpol: 0.9988
madonna: 0.9988
alice in chains: 0.9988
slipknot: 0.9988
travis: 0.9988
no doubt: 0.9988
abba: 0.9987


In [100]:
# 100th user embedding vector 
final_als_model.user_factors[99]

array([  6.352935  ,  17.518936  ,  -7.457646  ,  -5.548832  ,
        -0.39106834,   8.601491  ,   2.829964  ,   5.789644  ,
         0.06761858,  -0.60402834, -14.523184  ,  -4.6582084 ,
       -20.539946  ,  -2.5877166 ,   0.26064828,  18.011637  ,
         8.4733515 ,   9.190635  ,  -5.820763  ,  -1.2784431 ,
        -7.0106015 ,  -5.174522  ,   3.3742335 ,  -8.3755865 ,
        -2.5681212 ,  -1.7733479 , -14.005043  ,  11.805496  ,
         3.657497  ,  -6.7030344 ,  -6.8069406 ,   8.783989  ,
        -4.9753737 , -10.0220585 ,  14.069145  ,  -0.88645816,
         7.066891  ,  -4.668209  ,   1.211756  ,  16.354185  ,
        -4.995357  ,  -4.3557158 , -16.3567    ,   0.8134613 ,
         9.217     ,  10.820632  ,  -3.4508657 ,   8.3902    ,
       -10.172621  ,   3.324487  , -12.015239  ,   5.1850047 ,
         9.942706  ,  -5.7624245 ,   2.184448  ,  -7.635603  ,
         7.9484644 ,  -5.4894395 ,   8.486155  ,   6.0307503 ,
        20.604895  ,  -8.40745   ,  15.656537  ,  -2.69

## Make Recommendations

In [ ]:
# Select a specific user (by index position)
user_id = 99

# Get user's play history (artists and weighted counts) - model automatically excludes these from recommendations
recs = final_als_model.recommend(userid=user_id,
                                 user_items=user_plays_weighted[user_id],
                                 N=10,
                                 filter_already_liked_items=True)

# The recommend() method returns TWO separate arrays
artist_ids, scores = recs

# Loop through both arrays simultaneously
for artist_id, score in zip(artist_ids, scores):    # zip pairs them together
    print(f"{artists[artist_id]}: {score:.4f}")

jorge drexler: 1.6099
hercules and love affair: 1.4786
orishas: 1.4736
etta james: 1.4478
buraka som sistema: 1.3979
chromeo: 1.3898
babasónicos: 1.3632
calle 13: 1.3616
raphael: 1.3457
keren ann: 1.3419


## Exploring a User's Liked Artists

```python
# Full sparse row representation:
user_plays[user_id]
#   (artist_ID,    play_count)
#   (157195,       25.0)       ← Eminem
#   (3853,         50.0)       ← 50 Cent  
#   (12345,        10.0)       ← Dr. Dre

# .indices extracts ONLY the artist IDs:
liked_artists = user_plays[user_id].indices
# Result: [157195, 3853, 12345]

# .data would extract ONLY the play counts:
play_counts = user_plays[user_id].data
# Result: [25.0, 50.0, 10.0]

In [ ]:
# Select a specific user (by index position)
user_id = 100

# Get all artist IDs that this user has listened to (non-zero interactions)
liked_artists_id = user_plays[user_id].indices

# Loop through the first 10 artist IDs the user listened to
for artist_id in liked_artists_id[:10]:
    
    # Convert artist ID to actual name and print it
    print(artists[artist_id])

almodóvar, pedro
antonio carlos jobim & astrud gilberto
antony and the johnsons
beastie boys
beirut
benjamin biolay
best of
black eyed peas
christina rosenvinge
cocorosie


### Why ALS recommends Rock for Eminem fans

**ALS doesn't understand music genres.** It only finds patterns in listening behavior.

- **Eminem listeners** → also listen to **Incubus, The Killers** (rock bands)
- **Model assumption:** If users listen together, they're "similar"

### The Problem: Popularity Bias

Popular artists cluster together because **everyone** listens to them, regardless of genre. `(popularity bias)`

| You asked for | ALS gave you |
|--------------|--------------|
| Music that sounds like Rap | Music that Rap fans also listen to (Rock/Alternative) |

**ALS prioritizes co-occurrence ("fans also like") over audio features (genre).**

## Artist Recommendation Function

In [ ]:
def artists_you_may_also_like(artist_name: str, N: int = 10):
    """Recommend artists similar to a given artist based on embedding similarity."""

    # Convert artist name to its ID in the artists list
    artist_id = list(artists).index(artist_name)
    
    # Find top N similar artists using ALS item factors
    similar_ids, similar_scores = final_als_model.similar_items(itemid=artist_id, N=N)
    
    # Print each similar artist with their similarity score
    for id, score in zip(similar_ids, similar_scores):
        print(f"{artists[id]}: {score:.4f}")

artists_you_may_also_like("jay-z")

jay-z: 1.0000
outkast: 0.9975
kanye west: 0.9959
justin timberlake: 0.9953
gnarls barkley: 0.9949
hot chip: 0.9949
the postal service: 0.9947
prince: 0.9947
cypress hill: 0.9947
basement jaxx: 0.9946


## Find Artist ID Function

In [ ]:
def find_artist_id(artist_name: str):
    """Find and display the ID of an artist by their name."""
    try:
        artist_id = list(artists).index(artist_name.lower())
        print(f"{artists[artist_id]} --> {artist_id}")
        
    except ValueError:
        print(f"❌ Artist '{artist_name}' not found. Try checking spelling.")

find_artist_id("NaS")

nas --> 196221
